# **MODEL EXPERIMENTATION**
with GCP Integration


by Jack Phelan

In [75]:
#imports 
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)
from scripts.gcs_utils import (
    log_dataset_to_gcs,
    log_pipeline_run,
)
from kfp import compiler
from google.cloud import aiplatform

# Components
from vertex.components import (
    load_validate_data,
    split_data,
    oversample_training,
    fit_apply_preprocessing_v1,
    apply_preprocessing_v1,
    train_model,
    evaluate_model,
)

# Pipelines
from vertex.pipelines import preprocessing_pipeline, training_pipeline


In [76]:
# variable declarations
TARGET_COL = "readmission_within_30_days"
ID_COL = "patient_id"
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmissions_bucket_v2"


---
## KFP Preprocessing Pipeline

Self-contained KFP v2 components for the preprocessing pipeline.  
Order: `load_validate_data` → `split_data` → `oversample_training` → `fit_apply_preprocessing` → `apply_preprocessing`

In [77]:
# Components are defined in vertex/components/ and imported above:
#   load_validate_data                          ← ingest.py
#   split_data                                  ← split.py
#   oversample_training                         ← oversample.py
#   fit_apply_preprocessing_v1                  ← preprocessing.py
#   apply_preprocessing_v1                      ← preprocessing.py
#   train_model                                 ← train.py
#   evaluate_model                              ← evaluate.py


In [78]:
PREPROCESSING_PIPELINE_JSON = "../vertex/pipelines/readmissions_preprocessing_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=preprocessing_pipeline,
    package_path=PREPROCESSING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {PREPROCESSING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_preprocessing_pipeline.json


---
## KFP Training Pipeline

`train_model` and `evaluate_model` components extending the preprocessing pipeline.  
Order: `...preprocessing...` → `train_model` → `evaluate_model`

- **`model_type`**: `"logistic"` | `"random_forest"` | `"xgboost"`  
- **`hyperparams_json`**: JSON string of kwargs passed to the chosen estimator (e.g. `'{"n_estimators": 200, "max_depth": 5}'`)

In [79]:
TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path=TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {TRAINING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_training_pipeline.json


---
## Pipeline Submission with Vertex AI Datasets

Register the training CSV as a managed Vertex AI Dataset, then submit the training pipeline.  
The dataset resource name flows through `load_validate_data` metadata → Vertex ML Metadata, giving a native console lineage graph: **Dataset → Pipeline Run → Model**.

- To reuse an existing dataset instead of creating a new one, replace `TabularDataset.create(...)` with `aiplatform.TabularDataset(dataset_name="projects/.../datasets/...")`.

In [80]:
from pathlib import Path

RAW_TRAIN_PATH = Path("../data/raw/healthcare_readmissions_dataset_train.csv")
DATASET_VERSION = "v0.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Raw training data",
    tags=["raw"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/manifest.json


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v0-0


In [81]:
import time

# Register a Vertex AI Dataset for the current data version.
# This creates a managed dataset entry in Vertex AI linked to the GCS CSV.
DATASET_VERSION = "v0.0"
DATASET_GCS_URI = f"{BUCKET_ROOT_URI}/datasets/readmissions/{DATASET_VERSION}/train.csv"
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Reuse existing dataset if one already exists with this display name.
existing = aiplatform.TabularDataset.list(
    filter=f'display_name="readmissions-{DATASET_VERSION}"',
    order_by="create_time desc",
)

if existing:
    dataset = existing[0]
    print(f"Reusing existing dataset: {dataset.resource_name}")
else:
    # Create with simple retry for transient InternalServerError responses.
    for attempt in range(3):
        try:
            dataset = aiplatform.TabularDataset.create(
                display_name=f"readmissions-{DATASET_VERSION}",
                gcs_source=[DATASET_GCS_URI],
            )
            break
        except Exception as e:
            if attempt < 2:
                wait = 15 * (attempt + 1)
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

DATASET_RESOURCE_NAME = dataset.resource_name
print(f"Dataset registered : {DATASET_RESOURCE_NAME}")
print(f"GCS source         : {DATASET_GCS_URI}")


Reusing existing dataset: projects/182027088454/locations/us-central1/datasets/1513965389040582656
Dataset registered : projects/182027088454/locations/us-central1/datasets/1513965389040582656
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv


In [ ]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "readmissions-model-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

# Pass the managed Dataset resource name via input_artifacts so Vertex AI
# registers a real lineage edge: Dataset → Pipeline Run → Model.
job.submit(
    experiment=EXPERIMENT_NAME,
    input_artifacts={"training_dataset": DATASET_RESOURCE_NAME},
)
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260512132741?project=182027088454
Associating projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 to Experiment: readmissions-model-exp
Pipeline submitted: readmissions-training-v0.0-v0


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/readmissions-model-exp-pipeline-run-v0-20260512202819 to Experiment: readmissions-model-exp


Waiting for pipeline to complete...
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260512132741 current state:
3
PipelineJob run completed. Resource name

# **Uploading Dataset v1.0**
- removed outliers as defined in prep reports

In [84]:
raw_df = pd.read_csv(
    "../data/raw/healthcare_readmissions_dataset_train.csv",
    keep_default_na=False,
    na_values=[""],
)

raw_df.head()

,PatientID,Age,Gender,Ethnicity,Hospital ID,Height (m),Smoker,BMI,Weight (kg),Adjusted Weight (kg),Has Diabetes,Has Hypertension,Exercise Frequency,Diet Type,Number of Prior Visits,Medications Prescribed,Length of Stay,Type of Treatment,Readmission within 30 Days
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,0,0,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,0,0,Regular,High-fat,2.0,NaN,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,0,1,None,Other,NaN,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,0,0,None,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,0,0,Occasional,High-fat,7.0,5.0,7,Major Surgery,1


In [85]:
import scipy.stats as stats
import numpy as np
# age outlier checking
threshold = 3
age_z_scores = np.abs(stats.zscore(raw_df["Age"]))
age_outliers = np.where(age_z_scores > threshold)[0]
print(f"Identified {len(age_outliers)} age outliers at threshold {threshold}:")

display(raw_df.loc[age_outliers, "Age"])


Identified 80 age outliers at threshold 3:


122     158
136     167
253     126
535     147
547     120
       ... 
7251    136
7589    165
7814    195
7976    145
8027    156
Name: Age, Length: 80, dtype: int64

In [ ]:
df_transformed = raw_df.drop(index=age_outliers).reset_index(drop=True)

In [88]:
# looking into potential bmi outliers

threshold = 3.5
z_scores = np.abs(stats.zscore(df_transformed['BMI']))
bmi_outliers = np.where(z_scores > threshold)[0]
print(f"Identified {len(bmi_outliers)} BMI outliers at threshold {threshold}:")

display(df_transformed.loc[bmi_outliers, 'BMI'])

Identified 7 BMI outliers at threshold 3.5:


792     43.0
797     43.7
3878    44.0
4444     9.3
4771    43.5
4820    43.8
6398     8.3
Name: BMI, dtype: float64

In [89]:
df_transformed = df_transformed.copy().drop(index=bmi_outliers).reset_index(drop=True)

In [91]:
df_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 7951 entries, 0 to 7950
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PatientID                   7951 non-null   int64  
 1   Age                         7951 non-null   int64  
 2   Gender                      7951 non-null   str    
 3   Ethnicity                   7951 non-null   str    
 4   Hospital ID                 7951 non-null   str    
 5   Height (m)                  7951 non-null   float64
 6   Smoker                      7951 non-null   bool   
 7   BMI                         7951 non-null   float64
 8   Weight (kg)                 7951 non-null   float64
 9   Adjusted Weight (kg)        7951 non-null   float64
 10  Has Diabetes                7951 non-null   int64  
 11  Has Hypertension            7951 non-null   int64  
 12  Exercise Frequency          7951 non-null   str    
 13  Diet Type                   7951 non-null   

In [92]:
df_transformed.to_csv("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv", index=False)

In [ ]:

RAW_TRAIN_PATH = Path("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv")
DATASET_VERSION = "v1.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Training data without outliers",
    tags=["no_outliers"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v1.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v1.0/manifest.json


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/readmissions-model-exp-readmissions-data-v1-0 to Experiment: readmissions-model-exp


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v1-0


In [94]:
import time

# Register a Vertex AI Dataset for the current data version.
# This creates a managed dataset entry in Vertex AI linked to the GCS CSV.
DATASET_VERSION = "v1.0"
DATASET_GCS_URI = f"{BUCKET_ROOT_URI}/datasets/readmissions/{DATASET_VERSION}/train.csv"
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Reuse existing dataset if one already exists with this display name.
existing = aiplatform.TabularDataset.list(
    filter=f'display_name="readmissions-{DATASET_VERSION}"',
    order_by="create_time desc",
)

if existing:
    dataset = existing[0]
    print(f"Reusing existing dataset: {dataset.resource_name}")
else:
    # Create with simple retry for transient InternalServerError responses.
    for attempt in range(3):
        try:
            dataset = aiplatform.TabularDataset.create(
                display_name=f"readmissions-{DATASET_VERSION}",
                gcs_source=[DATASET_GCS_URI],
            )
            break
        except Exception as e:
            if attempt < 2:
                wait = 15 * (attempt + 1)
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

DATASET_RESOURCE_NAME = dataset.resource_name
print(f"Dataset registered : {DATASET_RESOURCE_NAME}")
print(f"GCS source         : {DATASET_GCS_URI}")


Creating TabularDataset
Create TabularDataset backing LRO: projects/182027088454/locations/us-central1/datasets/1453729744024502272/operations/2528011952219750400
TabularDataset created. Resource name: projects/182027088454/locations/us-central1/datasets/1453729744024502272
To use this TabularDataset in another session:
ds = aiplatform.TabularDataset('projects/182027088454/locations/us-central1/datasets/1453729744024502272')
Dataset registered : projects/182027088454/locations/us-central1/datasets/1453729744024502272
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v1.0/train.csv
